# Open-World Re-ID Evaluation (Video-to-Video & Image-to-Video)

This notebook evaluates distance matrices generated by models in the **Open-World** setting.
It computes and prints:
- **Rank-1 DIR @ FAR = 1%, 5%, 10%** with 95% Bootstrap Confidence Intervals
- **Rank-5 DIR @ FAR = 1%, 5%, 10%** with 95% Bootstrap Confidence Intervals


In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Add root directory to sys.path
CURRENT_DIR = Path.cwd()
ROOT_DIR = CURRENT_DIR if (CURRENT_DIR / "evaluation").exists() else CURRENT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))
if str(ROOT_DIR / "evaluation") not in sys.path:
    sys.path.append(str(ROOT_DIR / "evaluation"))

from evaluation.evaluation_utils import bootstrap_from_csv
print("Loaded evaluation_utils successfully!")
print(f"Project root: {ROOT_DIR}")


Loaded evaluation_utils successfully!
Project root: /d/hpc/projects/FRI/mm12755/DogReID-1553/DogReID-1553


In [2]:
# =====================================================================
# Configuration: Models and Candidate CSV Paths
# =====================================================================
MODELS = ["bot", "agw", "arbase", "sbs", "mgn", "tfclip", "baseline"]
MODALITIES = ["vid2vid", "img2vid"]

SEARCH_DIRS = [
    ROOT_DIR / "evaluation" / "csvs_open_results"
]

def find_open_csv(model_name: str, modality: str):
    """Find open-world distance matrix CSV for a given model and modality."""
    names = [model_name]
    if model_name == "baseline":
        names.extend(["dinov2", "dinov2_baseline"])

    for name in names:
        patterns = [
            f"{name}_{modality}/open_dist_matrix.csv",
            f"{name}_{modality}/vid2vid_open_dist_matrix.csv",
            f"{name}_{modality}/img2vid_open_dist_matrix.csv",
            f"{name}_open/open_dist_matrix.csv",
            f"{name}_open_{modality}/open_dist_matrix.csv",
        ]
        for search_dir in SEARCH_DIRS:
            if not search_dir.exists():
                continue
            for pat in patterns:
                candidate = search_dir / pat
                if candidate.exists() and candidate.stat().st_size > 1000:
                    return candidate
            # Direct folder check
            folder = search_dir / f"{name}_{modality}"
            if folder.exists():
                for f in folder.glob("*open*.csv"):
                    if f.stat().st_size > 1000:
                        return f
    return None

models_to_test = {}
for m in MODELS:
    for mod in MODALITIES:
        key = f"{m.upper()} ({mod})"
        csv_path = find_open_csv(m, mod)
        if csv_path:
            models_to_test[key] = str(csv_path)
            print(f"[FOUND] {key:<20} -> {csv_path.relative_to(ROOT_DIR)}")
        else:
            print(f"[PENDING] {key:<18} -> Not yet found in search directories")


[PENDING] BOT (vid2vid)      -> Not yet found in search directories
[PENDING] BOT (img2vid)      -> Not yet found in search directories
[PENDING] AGW (vid2vid)      -> Not yet found in search directories
[PENDING] AGW (img2vid)      -> Not yet found in search directories
[PENDING] ARBASE (vid2vid)   -> Not yet found in search directories
[PENDING] ARBASE (img2vid)   -> Not yet found in search directories
[PENDING] SBS (vid2vid)      -> Not yet found in search directories
[PENDING] SBS (img2vid)      -> Not yet found in search directories
[PENDING] MGN (vid2vid)      -> Not yet found in search directories
[PENDING] MGN (img2vid)      -> Not yet found in search directories
[PENDING] TFCLIP (vid2vid)   -> Not yet found in search directories
[PENDING] TFCLIP (img2vid)   -> Not yet found in search directories
[PENDING] BASELINE (vid2vid) -> Not yet found in search directories
[PENDING] BASELINE (img2vid) -> Not yet found in search directories


In [3]:
# =====================================================================
# Run Bootstrap Evaluation
# =====================================================================
M_ITERATIONS = 100
all_open_results = {}

for name, path in models_to_test.items():
    print("=" * 60)
    print(f"Evaluating {name} on {Path(path).name} ...")
    print("=" * 60)
    try:
        res = bootstrap_from_csv(path, m=M_ITERATIONS, mode="open")
        all_open_results[name] = res
    except Exception as e:
        print(f"[ERROR] Failed to evaluate {name}: {e}")


In [4]:
# =====================================================================
# Display Open-World Benchmark Table
# =====================================================================
rows = []
for name, res in all_open_results.items():
    full_fars = res["full_set"]["fars"]
    
    col_dict = {"Model (Modality)": name}
    for target, label in [(0.01, "FAR=1%"), (0.05, "FAR=5%"), (0.10, "FAR=10%")]:
        idx = np.argmin(np.abs(full_fars - target))
        
        # Rank-1 DIR
        r1_val = res["full_set"]["dirs_r1"][idx] * 100
        r1_l = res["dirs_r1_ci"][0][idx] * 100
        r1_u = res["dirs_r1_ci"][1][idx] * 100
        col_dict[f"R1 @ {label}"] = f"{r1_val:.2f} [{r1_l:.2f}-{r1_u:.2f}]"
        
        # Rank-5 DIR
        r5_val = res["full_set"]["dirs_r5"][idx] * 100
        r5_l = res.get("dirs_r5_ci", res["dirs_r1_ci"])[0][idx] * 100
        r5_u = res.get("dirs_r5_ci", res["dirs_r1_ci"])[1][idx] * 100
        col_dict[f"R5 @ {label}"] = f"{r5_val:.2f} [{r5_l:.2f}-{r5_u:.2f}]"
        
    rows.append(col_dict)

df_open = pd.DataFrame(rows)

print("=" * 125)
print(f"{"Model (Modality)":<22} | {"R1 @ FAR=1%":<24} | {"R1 @ FAR=5%":<24} | {"R1 @ FAR=10%":<24}")
print("=" * 125)
for r in rows:
    print(f"{r["Model (Modality)"]:<22} | {r["R1 @ FAR=1%"]:<24} | {r["R1 @ FAR=5%"]:<24} | {r["R1 @ FAR=10%"]:<24}")
print("-" * 125)

df_open


Model (Modality)       | R1 @ FAR=1%              | R1 @ FAR=5%              | R1 @ FAR=10%            
-----------------------------------------------------------------------------------------------------------------------------


""
